# TinyVLM v2 -- SigLIP2 + Qwen2.5-0.5B

Rebuild of the pipeline after diagnosing why run 1 answered with one or two
fixed responses regardless of the image.

## What went wrong in run 1

| Root cause | Evidence | Fix |
|---|---|---|
| Projector alignment corpus ~350x too small | `--max_examples 8000` on `captions.txt` = ~1600 distinct images (5 captions per image, stored consecutively); 500 optimizer steps total | Flickr8k + Flickr30k, ~39k images / ~185k pairs |
| The failure was invisible | `loss = loss/accum_steps` was assigned *before* `set_postfix`, so the logged Stage-1 `0.786` was really ~3.14 -- about the text-only prior | logging reports the true per-token loss |
| Trained to emit exactly one word | VQAv2 targets tokenize to `['red', '<\|im_end\|>']` -- 2 supervised tokens out of 238, 10k times | mix VQA + detailed captions; VQA rows carry an explicit length hint |
| Visual tokens outside the embedding manifold | SigLIP token norm 47.2 vs Qwen embedding norm 0.45 (~104x) | projector ends in a LayerNorm initialised to the LLM's own embedding std -> 1.01x |
| LoRA never touched the MLP block | `target_modules=[q,k,v,o]` | + `gate_proj, up_proj, down_proj` |
| 1.1 GB adapter | adding `<image>` forced `resize_token_embeddings`, so PEFT serialised the whole embedding matrix | reuse Qwen's existing `<\|image_pad\|>`; adapter ~35 MB |
| GPU starved at 1.33 it/s | `num_workers=0` + full-resolution decode every epoch | prep-time resize, worker processes, fp16 AMP |
| No way to notice before upload | no held-out loss, no sample generations | eval + generations every 500 steps, plus a grounding gate between stages |

## Session setup

- **Accelerator:** GPU T4 x2
- **Internet:** On
- **Add data:** the Kaggle dataset `adityajn105/flickr8k`

Rough budget: 1.5-2 h data prep, ~2 h Stage 1, ~1.5 h Stage 2.

> The `%%writefile` cells below are generated from the training repo by
> `tools/build_notebook.py`. Edit the repo and regenerate -- don't hand-edit
> them here, which is how run 1's notebook and repo copies drifted apart.

## 1. Environment

In [ ]:
!pip install -q "transformers>=4.45" "peft>=0.11" accelerate datasets huggingface_hub

In [ ]:
import torch, transformers, peft
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__, "| peft", peft.__version__)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NO GPU -- set Accelerator to GPU T4 x2 in the sidebar")

## 2. Project files

In [ ]:
import os
os.chdir("/kaggle/working")
print(os.getcwd())

In [ ]:
%%writefile model.py
"""
model.py

TinyVLM architecture:
  SigLIP2 vision encoder (frozen) -> pixel-shuffle -> MLP projector -> Qwen2.5 LLM (LoRA)

Design notes / differences from the first version, all of which mattered:

1.  We reuse Qwen2.5's existing `<|image_pad|>` token instead of adding a new
    `<image>` token. Adding a token forced `resize_token_embeddings(151666)`,
    which actually *shrank* the embedding matrix (Qwen ships 151936 rows, the
    tokenizer only knows 151665) and forced PEFT to serialize the whole
    embedding table into the adapter -- a 1.1 GB adapter instead of ~9 MB.

2.  The image placeholder is repeated `num_visual_tokens` times in `input_ids`
    rather than appearing once and being spliced. That keeps `input_ids`,
    `labels` and `attention_mask` the same length as `inputs_embeds`, so
    padding and masking are handled by ordinary collation instead of bespoke
    index arithmetic.

3.  The projector ends in a LayerNorm whose gain is initialised to the LLM's
    own token-embedding std. Raw SigLIP hidden states are ~10-20x larger in
    norm than Qwen token embeddings; without this the visual tokens land far
    outside the embedding manifold and the projector burns its (short)
    training budget just learning to rescale.

4.  Pixel-shuffle folds each 2x2 patch neighbourhood into one token
    (196 -> 49 tokens, no information discarded). With a 0.5B LLM and a
    single-session training budget, sample efficiency beats spatial
    resolution -- and it makes the supervised-token / total-token ratio
    roughly 4x better.
"""

import torch
import torch.nn as nn
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, AutoImageProcessor

VISION_MODEL_NAME = "google/siglip2-base-patch16-224"
LLM_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Already present in the Qwen2.5 vocabulary (reserved for Qwen2-VL), so using it
# needs no vocab resize and no embedding surgery.
IMAGE_TOKEN = "<|image_pad|>"


def _load_vision_tower(name: str, dtype):
    """
    Return just the ViT stack.

    Deliberately routed through AutoModel so the architecture comes from the
    checkpoint's own config. Hand-picking a class is a trap: despite the name,
    `google/siglip2-base-patch16-224` is stored as `siglip_vision_model`, and
    loading it with `Siglip2VisionModel` does NOT raise -- it reports
    "Reinit due to size mismatch" for `embeddings.patch_embedding` and
    `embeddings.position_embedding` and hands back an encoder whose input layer
    is randomly initialised. That produces a vision tower that emits noise, and
    nothing downstream would tell you.

    Keeping only the returned submodule makes the parent (and SigLIP's unused
    text tower) unreachable, so it is freed on the next collection anyway.
    """
    return AutoModel.from_pretrained(name, dtype=dtype).vision_model


def _assert_vision_tower_loaded(vision_encoder, name: str):
    """Cheap guard against a silently reinitialised patch embedding (see
    `_load_vision_tower`). A real conv patch embedding is 4-D [D, 3, P, P];
    the reinit path leaves a 2-D weight behind."""
    w = getattr(getattr(vision_encoder, "embeddings", None), "patch_embedding", None)
    if w is None:
        return
    w = w.weight
    if w.dim() != 4 or w.shape[1] != 3:
        raise RuntimeError(
            f"vision tower for {name!r} did not load its pretrained patch embedding "
            f"(got weight shape {tuple(w.shape)}, expected [D, 3, P, P]). The encoder "
            "would emit noise. Check the transformers version / checkpoint pairing."
        )


def pixel_shuffle(x: torch.Tensor, stride: int) -> torch.Tensor:
    """
    [B, N, D] -> [B, N/stride^2, D*stride^2], folding each stride x stride
    neighbourhood of patches into a single token. Unlike pooling this is
    lossless -- the channel dim absorbs what the sequence dim gives up.
    """
    if stride == 1:
        return x
    b, n, d = x.shape
    h = w = int(n**0.5)
    if h * w != n:
        raise ValueError(f"expected a square patch grid, got {n} patches")
    if h % stride or w % stride:
        raise ValueError(f"{h}x{w} patch grid is not divisible by stride {stride}")

    x = x.view(b, h, w // stride, d * stride)
    x = x.permute(0, 2, 1, 3).contiguous()
    x = x.view(b, w // stride, h // stride, d * stride * stride)
    x = x.permute(0, 2, 1, 3).contiguous()
    return x.view(b, (h // stride) * (w // stride), d * stride * stride)


class Projector(nn.Module):
    """Maps vision embedding dim -> LLM embedding dim, at the LLM's own scale."""

    def __init__(self, vision_dim: int, llm_dim: int, pool_stride: int = 2):
        super().__init__()
        self.pool_stride = pool_stride
        in_dim = vision_dim * pool_stride * pool_stride
        self.net = nn.Sequential(
            nn.Linear(in_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
        )
        # Gain is re-initialised to the LLM embedding std by TinyVLM.__init__.
        self.out_norm = nn.LayerNorm(llm_dim)

    def forward(self, x):
        x = pixel_shuffle(x, self.pool_stride)
        return self.out_norm(self.net(x))


class TinyVLM(nn.Module):
    def __init__(
        self,
        vision_model_name: str = VISION_MODEL_NAME,
        llm_model_name: str = LLM_MODEL_NAME,
        freeze_vision: bool = True,
        freeze_llm: bool = True,
        pool_stride: int = 2,
        dtype: torch.dtype = torch.float32,
    ):
        super().__init__()

        # --- Vision encoder ---
        # Load the vision tower alone. `AutoModel(...).vision_model` also
        # materialises SigLIP's unused text tower (~450 MB) and emits confusing
        # vocab warnings from its CLIP-inherited text config.
        self.vision_encoder = _load_vision_tower(vision_model_name, dtype)
        self.image_processor = AutoImageProcessor.from_pretrained(vision_model_name)
        vision_cfg = self.vision_encoder.config
        vision_dim = vision_cfg.hidden_size
        _assert_vision_tower_loaded(self.vision_encoder, vision_model_name)

        # --- LLM ---
        self.tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.llm = AutoModelForCausalLM.from_pretrained(llm_model_name, dtype=dtype)
        llm_dim = self.llm.config.hidden_size

        self.image_token_id = self.tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)
        if self.image_token_id is None or self.image_token_id == self.tokenizer.unk_token_id:
            raise RuntimeError(
                f"{IMAGE_TOKEN!r} is not in the {llm_model_name} vocabulary. "
                "Pick another reserved token rather than adding one -- see the "
                "module docstring for why adding one is a trap."
            )

        # --- Projector ---
        self.pool_stride = pool_stride
        grid = vision_cfg.image_size // vision_cfg.patch_size
        self.num_visual_tokens = (grid // pool_stride) ** 2

        self.projector = Projector(vision_dim, llm_dim, pool_stride).to(dtype)

        # Match the LLM's embedding scale so visual tokens start life inside the
        # distribution the LLM was trained on.
        with torch.no_grad():
            emb = self.llm.get_input_embeddings().weight
            # Rows beyond the tokenizer's range are unused padding in Qwen and
            # are near-zero; they would skew the statistic.
            target_std = emb[: len(self.tokenizer)].float().std().item()
            self.projector.out_norm.weight.fill_(target_std)
            self.projector.out_norm.bias.zero_()
        self.embed_std = target_std

        if freeze_vision:
            for p in self.vision_encoder.parameters():
                p.requires_grad = False
        if freeze_llm:
            for p in self.llm.parameters():
                p.requires_grad = False

    # ------------------------------------------------------------------ #

    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """[B, 3, H, W] -> visual tokens [B, num_visual_tokens, llm_dim]"""
        # When the encoder is frozen there is nothing to backprop into it, so
        # skipping its activation graph is a large memory and speed win.
        vision_frozen = not any(p.requires_grad for p in self.vision_encoder.parameters())
        with torch.set_grad_enabled(not vision_frozen and torch.is_grad_enabled()):
            vision_out = self.vision_encoder(pixel_values=pixel_values).last_hidden_state
        return self.projector(vision_out)

    def build_inputs_embeds(self, input_ids: torch.Tensor, pixel_values: torch.Tensor) -> torch.Tensor:
        """
        Replace every `<|image_pad|>` embedding with the corresponding projected
        visual token. Sequence length is unchanged, so labels and attention_mask
        built by the collator line up by construction.
        """
        inputs_embeds = self.llm.get_input_embeddings()(input_ids)

        if pixel_values is None:
            return inputs_embeds

        visual = self.encode_image(pixel_values)  # [B, N, D]
        mask = input_ids == self.image_token_id

        expected = visual.shape[0] * visual.shape[1]
        found = int(mask.sum())
        if found != expected:
            raise ValueError(
                f"expected {expected} image placeholder tokens "
                f"({visual.shape[0]} images x {visual.shape[1]} tokens) but found {found}. "
                "The dataset and the model disagree on num_visual_tokens -- check pool_stride."
            )

        return inputs_embeds.masked_scatter(
            mask.unsqueeze(-1), visual.to(inputs_embeds.dtype)
        )

    def forward(self, input_ids, attention_mask, pixel_values=None, labels=None):
        inputs_embeds = self.build_inputs_embeds(input_ids, pixel_values)
        return self.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
        )

    @torch.no_grad()
    def generate(self, input_ids, attention_mask, pixel_values=None, max_new_tokens=128, **gen_kwargs):
        inputs_embeds = self.build_inputs_embeds(input_ids, pixel_values)
        gen_kwargs.setdefault("pad_token_id", self.tokenizer.pad_token_id)
        gen_kwargs.setdefault("eos_token_id", self.tokenizer.eos_token_id)
        # With inputs_embeds (and no input_ids) HF returns ONLY the new tokens.
        return self.llm.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            **gen_kwargs,
        )

    # ------------------------------------------------------------------ #

    def trainable_parameter_report(self) -> str:
        total = sum(p.numel() for p in self.parameters())
        train = sum(p.numel() for p in self.parameters() if p.requires_grad)
        by_part = {}
        for name, mod in [
            ("vision", self.vision_encoder),
            ("projector", self.projector),
            ("llm", self.llm),
        ]:
            by_part[name] = sum(p.numel() for p in mod.parameters() if p.requires_grad)
        parts = ", ".join(f"{k}={v:,}" for k, v in by_part.items())
        return f"trainable {train:,} / {total:,} ({100 * train / total:.3f}%)  [{parts}]"

In [ ]:
%%writefile dataset.py
"""
dataset.py

JSONL schema (LLaVA-style, one JSON object per line):

  {"image": "images/0001.jpg",
   "conversations": [{"from": "human", "value": "<image>\nWhat is the man doing?"},
                     {"from": "gpt",   "value": "He is riding a bicycle."}]}

The literal marker `<image>` inside a human turn is expanded to
`num_visual_tokens` copies of the model's real image placeholder token before
tokenisation. If no marker is present it is prepended to the first human turn.

Two things here fix concrete failures in the first version:

*   Loss is computed on assistant turns only, located by tokenising growing
    conversation prefixes with `apply_chat_template`. This is version-proof --
    it never hardcodes Qwen's `<|im_start|>` layout -- and it supports
    multi-turn conversations, which the old question/answer schema could not.

*   `SHORT_ANSWER_HINT` is appended to VQA-style questions at data-prep time
    (the LLaVA-1.5 trick). Without it, training on VQAv2's one-word answers
    teaches the model that *every* reply is one word, and it permanently loses
    the ability to write a sentence. With it, answer length becomes something
    the prompt controls at inference time.
"""

import json
import random
from pathlib import Path

import torch
from PIL import Image
from torch.utils.data import Dataset

IMAGE_MARKER = "<image>"

# Must be identical at train and inference time -- import it, don't retype it.
SYSTEM_PROMPT = "You are a helpful assistant that can see and understand images."

SHORT_ANSWER_HINT = "\nAnswer the question using a single word or phrase."

DESCRIBE_INSTRUCTIONS = [
    "Describe this image.",
    "What is happening in this image?",
    "Write a short caption for this image.",
    "Describe the image in detail.",
    "What do you see in this picture?",
    "Provide a description of this image.",
]


class VLMDataset(Dataset):
    def __init__(
        self,
        jsonl_path: str,
        image_root: str,
        tokenizer,
        image_processor,
        num_visual_tokens: int,
        image_token: str,
        max_length: int = 1024,
    ):
        self.examples = [
            json.loads(l) for l in Path(jsonl_path).read_text(encoding="utf-8").splitlines() if l.strip()
        ]
        self.image_root = Path(image_root)
        self.tokenizer = tokenizer
        self.image_processor = image_processor
        self.max_length = max_length
        self.image_token = image_token
        self.image_placeholder = image_token * num_visual_tokens

    def __len__(self):
        return len(self.examples)

    def _to_messages(self, ex):
        turns = ex["conversations"]
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        seen_image = False
        for t in turns:
            role = "user" if t["from"] in ("human", "user") else "assistant"
            content = t["value"]
            if role == "user":
                if IMAGE_MARKER in content:
                    content = content.replace(IMAGE_MARKER, self.image_placeholder)
                    seen_image = True
                elif not seen_image:
                    content = f"{self.image_placeholder}\n{content}"
                    seen_image = True
            messages.append({"role": role, "content": content})
        return messages

    def _encode(self, text, **kw):
        return self.tokenizer(text, add_special_tokens=False, **kw)["input_ids"]

    def __getitem__(self, idx):
        ex = self.examples[idx]

        image = Image.open(self.image_root / ex["image"]).convert("RGB")
        pixel_values = self.image_processor(images=image, return_tensors="pt")["pixel_values"][0]

        messages = self._to_messages(ex)

        input_ids: list[int] = []
        labels: list[int] = []
        prev_len = 0

        for i, msg in enumerate(messages):
            if msg["role"] != "assistant":
                continue
            # Everything up to and including the "<|im_start|>assistant\n" header.
            prompt_text = self.tokenizer.apply_chat_template(
                messages[:i], add_generation_prompt=True, tokenize=False
            )
            full_text = self.tokenizer.apply_chat_template(
                messages[: i + 1], add_generation_prompt=False, tokenize=False
            )
            prompt_ids = self._encode(prompt_text)
            full_ids = self._encode(full_text)

            # Context since the previous assistant turn is supervised as -100,
            # the assistant's own tokens carry the loss.
            input_ids.extend(full_ids[prev_len:])
            labels.extend([-100] * (len(prompt_ids) - prev_len))
            labels.extend(full_ids[len(prompt_ids):])
            prev_len = len(full_ids)

        input_ids = input_ids[: self.max_length]
        labels = labels[: self.max_length]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "pixel_values": pixel_values,
            "n_supervised": int(sum(1 for l in labels if l != -100)),
        }


def make_collate_fn(pad_token_id: int):
    """Right-padded collation. Causal attention + right padding means the pad
    positions cannot influence any real token, and they are masked out of the
    loss and the attention mask regardless."""

    def collate_fn(batch):
        # Drop examples whose answer was entirely truncated away; a batch with
        # zero supervised tokens produces a NaN loss.
        batch = [b for b in batch if b["n_supervised"] > 0] or batch[:1]

        max_len = max(b["input_ids"].size(0) for b in batch)
        n = len(batch)

        input_ids = torch.full((n, max_len), pad_token_id, dtype=torch.long)
        labels = torch.full((n, max_len), -100, dtype=torch.long)
        attention_mask = torch.zeros((n, max_len), dtype=torch.long)

        for i, b in enumerate(batch):
            L = b["input_ids"].size(0)
            input_ids[i, :L] = b["input_ids"]
            labels[i, : b["labels"].size(0)] = b["labels"]
            attention_mask[i, :L] = 1

        return {
            "input_ids": input_ids,
            "labels": labels,
            "attention_mask": attention_mask,
            "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        }

    return collate_fn


def build_prompt(tokenizer, question: str, image_placeholder: str, short_answer: bool = False) -> str:
    """Single source of truth for inference-time prompt construction."""
    if short_answer:
        question = question.rstrip() + SHORT_ANSWER_HINT
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{image_placeholder}\n{question}"},
    ]
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)


def random_describe_instruction(rng: random.Random | None = None) -> str:
    return (rng or random).choice(DESCRIBE_INSTRUCTIONS)

In [ ]:
%%writefile prepare_data.py
"""
prepare_data.py

Builds the JSONL corpora consumed by dataset.VLMDataset.

Why the corpus changed
----------------------
The first run aligned the projector on `--max_examples 8000` rows of Flickr8k's
captions.txt. That file stores 5 captions per image consecutively, so 8000 rows
is only ~1600 distinct images, and the projector never saw enough visual
variety to learn anything. LLaVA's equivalent stage uses 558k pairs.

Stage 1 here uses Flickr8k + Flickr30k: ~39k distinct images / ~195k caption
pairs. That is ~24x more images, and it fits comfortably in one Kaggle session.

Stage 2 mixes three response styles so the model learns that answer length is
controlled by the prompt rather than baked in:
  * vqav2    -- one-word answers, tagged with the short-answer hint
  * recap    -- detailed multi-sentence descriptions
  * captions -- one-sentence descriptions

Images are resized at prep time (short side 256, JPEG q90). The old pipeline
symlinked full-resolution originals and re-decoded them every epoch, which is a
large part of why Stage 1 only managed 1.33 it/s.

Usage:
    python prepare_data.py --task flickr8k  --kaggle_input_dir /kaggle/input/... \
        --out_jsonl data/cap8k.jsonl --out_image_dir data/images --max_examples 40000
    python prepare_data.py --task flickr30k --out_jsonl data/cap30k.jsonl --out_image_dir data/images
    python prepare_data.py --task vqav2     --out_jsonl data/vqa.jsonl   --out_image_dir data/images --max_examples 30000
    python prepare_data.py --task recap     --out_jsonl data/recap.jsonl --out_image_dir data/images --max_examples 20000
    python prepare_data.py --task mix --inputs data/vqa.jsonl data/recap.jsonl data/cap8k.jsonl \
        --out_jsonl data/stage2.jsonl
"""

import argparse
import csv
import json
import random
import sys
from pathlib import Path

from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parent.parent / "Model"))
sys.path.insert(0, str(Path(__file__).resolve().parent))

try:
    from dataset import SHORT_ANSWER_HINT, DESCRIBE_INSTRUCTIONS, IMAGE_MARKER
except ImportError:  # the notebook writes all the .py files flat
    SHORT_ANSWER_HINT = "\nAnswer the question using a single word or phrase."
    IMAGE_MARKER = "<image>"
    DESCRIBE_INSTRUCTIONS = [
        "Describe this image.",
        "What is happening in this image?",
        "Write a short caption for this image.",
        "Describe the image in detail.",
        "What do you see in this picture?",
        "Provide a description of this image.",
    ]

TARGET_SHORT_SIDE = 256
JPEG_QUALITY = 90


def save_resized(img: Image.Image, dst: Path) -> bool:
    """Resize so the short side is TARGET_SHORT_SIDE, save JPEG.
    Returns False if the file already existed."""
    if dst.exists():
        return False
    img = img.convert("RGB")
    w, h = img.size
    scale = TARGET_SHORT_SIDE / min(w, h)
    if scale < 1.0:
        img = img.resize((max(1, round(w * scale)), max(1, round(h * scale))), Image.BICUBIC)
    dst.parent.mkdir(parents=True, exist_ok=True)
    img.save(dst, "JPEG", quality=JPEG_QUALITY)
    return True


def turn(human: str, gpt: str):
    return [
        {"from": "human", "value": f"{IMAGE_MARKER}\n{human}"},
        {"from": "gpt", "value": gpt},
    ]


def write_jsonl(records, out_jsonl: str):
    Path(out_jsonl).parent.mkdir(parents=True, exist_ok=True)
    with open(out_jsonl, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Wrote {len(records):,} examples -> {out_jsonl}")


# --------------------------------------------------------------------------- #
# Flickr8k (Kaggle input)
# --------------------------------------------------------------------------- #

def _find_flickr8k_files(root: Path):
    caps = list(root.rglob("captions.txt")) or list(root.rglob("Flickr8k.token.txt"))
    imgs = [d for d in root.rglob("Images") if d.is_dir()] or [d for d in root.rglob("images") if d.is_dir()]
    if not caps or not imgs:
        tree = "\n".join(str(p) for p in sorted(root.rglob("*"))[:50])
        raise FileNotFoundError(
            f"Could not find captions file and/or Images dir under {root}.\n"
            f"First 50 entries:\n{tree}"
        )
    return caps[0], imgs[0]


def prepare_flickr8k(kaggle_input_dir, out_jsonl, out_image_dir, max_examples=None, seed=0):
    rng = random.Random(seed)
    captions_file, images_dir = _find_flickr8k_files(Path(kaggle_input_dir))
    print(f"captions: {captions_file}\nimages:   {images_dir}")

    out_image_dir = Path(out_image_dir)
    records, saved = [], 0

    with open(captions_file, "r", encoding="utf-8") as f:
        head = f.readline()
        f.seek(0)
        reader = csv.reader(f)
        if head.strip().lower().startswith("image"):
            next(reader)
        for row in reader:
            if len(row) < 2:
                continue
            name, caption = row[0].split("#")[0], ",".join(row[1:]).strip()
            src = images_dir / name
            if not src.exists() or not caption:
                continue
            dst_name = f"f8k_{name}"
            try:
                if save_resized(Image.open(src), out_image_dir / dst_name):
                    saved += 1
            except Exception:
                continue
            records.append(
                {"image": dst_name, "conversations": turn(rng.choice(DESCRIBE_INSTRUCTIONS), caption)}
            )
            if max_examples and len(records) >= max_examples:
                break

    if not records:
        raise RuntimeError("0 Flickr8k records matched -- captions and images dirs don't correspond.")
    print(f"  saved {saved:,} new images")
    write_jsonl(records, out_jsonl)


# --------------------------------------------------------------------------- #
# Flickr30k (Hugging Face, images embedded in parquet)
# --------------------------------------------------------------------------- #

def prepare_flickr30k(out_jsonl, out_image_dir, max_examples=None, seed=0, captions_per_image=5):
    from datasets import load_dataset

    rng = random.Random(seed)
    out_image_dir = Path(out_image_dir)

    # nlphuji/flickr30k ships everything under a single split (named "test"),
    # with a per-row `split` column carrying the real train/val/test label.
    ds = load_dataset("nlphuji/flickr30k", split="test", streaming=True)

    records, n_img = [], 0
    for ex in ds:
        if ex.get("split") == "test":
            continue  # keep the official test images out of training
        name = f"f30k_{ex.get('img_id', n_img)}.jpg"
        try:
            save_resized(ex["image"], out_image_dir / name)
        except Exception:
            continue
        n_img += 1
        caps = ex["caption"]
        if isinstance(caps, str):
            caps = [caps]
        for c in caps[:captions_per_image]:
            c = c.strip()
            if c:
                records.append(
                    {"image": name, "conversations": turn(rng.choice(DESCRIBE_INSTRUCTIONS), c)}
                )
        if max_examples and len(records) >= max_examples:
            break

    print(f"  {n_img:,} images -> {len(records):,} caption pairs")
    write_jsonl(records, out_jsonl)


# --------------------------------------------------------------------------- #
# VQAv2 (short answers)
# --------------------------------------------------------------------------- #

def prepare_vqav2(out_jsonl, out_image_dir, max_examples=30000, split="validation"):
    """lmms-lab/VQAv2 is a script-free parquet mirror with embedded images.
    Only validation/testdev/test exist; validation is the one with answers."""
    from datasets import load_dataset

    out_image_dir = Path(out_image_dir)
    ds = load_dataset("lmms-lab/VQAv2", split=split, streaming=True)

    records = []
    for i, ex in enumerate(ds):
        if i >= max_examples:
            break
        name = f"vqa_{i:06d}.jpg"
        try:
            save_resized(ex["image"], out_image_dir / name)
        except Exception:
            continue
        answer = ex.get("multiple_choice_answer") or ex["answers"][0]["answer"]
        question = ex["question"].strip()
        records.append(
            {
                "image": name,
                # The hint is what stops one-word supervision from destroying the
                # model's ability to produce sentences.
                "conversations": turn(question + SHORT_ANSWER_HINT, str(answer).strip()),
                "style": "short",
            }
        )

    write_jsonl(records, out_jsonl)


# --------------------------------------------------------------------------- #
# LLaVA-ReCap (detailed descriptions)
# --------------------------------------------------------------------------- #

def prepare_recap(out_jsonl, out_image_dir, max_examples=20000, max_words=140):
    """lmms-lab/LLaVA-ReCap-118K: image + a 2-turn conversation whose assistant
    reply is a detailed multi-sentence description. This is what teaches the
    model to write prose instead of a single word."""
    from datasets import load_dataset

    out_image_dir = Path(out_image_dir)
    ds = load_dataset("lmms-lab/LLaVA-ReCap-118K", split="train", streaming=True)

    records = []
    for i, ex in enumerate(ds):
        if len(records) >= max_examples:
            break
        name = f"recap_{i:06d}.jpg"
        try:
            save_resized(ex["image"], out_image_dir / name)
        except Exception:
            continue

        convs = []
        for t in ex["conversations"]:
            val = t["value"].strip()
            if t["from"] in ("gpt", "assistant"):
                words = val.split()
                if len(words) > max_words:  # keep sequence lengths bounded
                    val = " ".join(words[:max_words]).rstrip(",;: ") + "."
            convs.append({"from": t["from"], "value": val})

        if not any(t["from"] in ("gpt", "assistant") and t["value"] for t in convs):
            continue
        records.append({"image": name, "conversations": convs, "style": "long"})

    write_jsonl(records, out_jsonl)


# --------------------------------------------------------------------------- #
# Mix
# --------------------------------------------------------------------------- #

def mix(inputs, out_jsonl, seed=0, caps=None):
    rng = random.Random(seed)
    all_recs = []
    for idx, path in enumerate(inputs):
        recs = [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]
        limit = caps[idx] if caps and idx < len(caps) and caps[idx] > 0 else len(recs)
        rng.shuffle(recs)
        recs = recs[:limit]
        print(f"  {path}: {len(recs):,}")
        all_recs.extend(recs)
    rng.shuffle(all_recs)
    write_jsonl(all_recs, out_jsonl)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--task", choices=["flickr8k", "flickr30k", "vqav2", "recap", "mix"], required=True)
    p.add_argument("--kaggle_input_dir", default="/kaggle/input/flickr8k")
    p.add_argument("--out_jsonl", required=True)
    p.add_argument("--out_image_dir", default="data/images")
    p.add_argument("--max_examples", type=int, default=None)
    p.add_argument("--inputs", nargs="*", default=[], help="mix: input jsonl paths")
    p.add_argument("--caps", nargs="*", type=int, default=None, help="mix: per-input row cap")
    p.add_argument("--seed", type=int, default=0)
    a = p.parse_args()

    if a.task == "flickr8k":
        prepare_flickr8k(a.kaggle_input_dir, a.out_jsonl, a.out_image_dir, a.max_examples, a.seed)
    elif a.task == "flickr30k":
        prepare_flickr30k(a.out_jsonl, a.out_image_dir, a.max_examples, a.seed)
    elif a.task == "vqav2":
        prepare_vqav2(a.out_jsonl, a.out_image_dir, a.max_examples or 30000)
    elif a.task == "recap":
        prepare_recap(a.out_jsonl, a.out_image_dir, a.max_examples or 20000)
    else:
        mix(a.inputs, a.out_jsonl, a.seed, a.caps)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train.py
"""
train.py -- both stages, one script.

    Stage 1  projector alignment    (vision frozen, LLM frozen, projector trains)
    Stage 2  instruction tuning     (vision frozen, LoRA on LLM, projector fine-tunes)

The previous code had a separate script per stage, and the copies in the repo
and in the notebook had already drifted apart (the notebook's stage-1 script had
gradient accumulation, the repo's did not). One file, one `--stage` flag.

Fixes relative to the first version
-----------------------------------
* Loss logging was wrong: `loss = loss / accum_steps` was assigned before
  `set_postfix(loss=loss.item())`, so the reported Stage-1 loss of 0.786 was
  really ~3.14. Here the reported number is always the true per-token loss.
* `num_workers=0` meant image decode was single-threaded and the GPU starved
  (1.33 it/s for a 0.5B model). Now workers + prefetch + persistent workers.
* No LR schedule and no warmup. Now warmup + cosine.
* No gradient clipping, no AMP. Now both (fp16 -- T4 is Turing, no bf16).
* No held-out loss and no sample generations, so a collapsed projector was
  invisible until after the upload. Now both, every `--eval_every` steps.

Usage:
    python train.py --stage 1 --data data/stage1.jsonl --image_root data/images \
        --output_dir ckpt/stage1 --epochs 1 --batch_size 16 --lr 1e-3

    python train.py --stage 2 --data data/stage2.jsonl --image_root data/images \
        --projector_ckpt ckpt/stage1/projector.pt --output_dir ckpt/stage2 \
        --epochs 2 --batch_size 8 --lr 2e-4
"""

import argparse
import json
import math
import os
import random
import sys
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

for _p in (Path(__file__).resolve().parent.parent / "Model", Path(__file__).resolve().parent):
    sys.path.insert(0, str(_p))

from model import TinyVLM, IMAGE_TOKEN            # noqa: E402
from dataset import VLMDataset, make_collate_fn, build_prompt   # noqa: E402


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--stage", type=int, choices=[1, 2], required=True)
    p.add_argument("--data", required=True)
    p.add_argument("--image_root", required=True)
    p.add_argument("--output_dir", required=True)
    p.add_argument("--projector_ckpt", default=None, help="stage 2: stage-1 projector")

    p.add_argument("--epochs", type=int, default=1)
    p.add_argument("--batch_size", type=int, default=16)
    p.add_argument("--accum_steps", type=int, default=2)
    p.add_argument("--lr", type=float, default=None, help="default: 1e-3 (stage 1) / 2e-4 (stage 2)")
    p.add_argument("--projector_lr", type=float, default=2e-5, help="stage 2 only")
    p.add_argument("--warmup_ratio", type=float, default=0.03)
    p.add_argument("--weight_decay", type=float, default=0.0)
    p.add_argument("--max_grad_norm", type=float, default=1.0)
    p.add_argument("--max_steps", type=int, default=-1, help="hard budget cap on optimizer steps")

    p.add_argument("--pool_stride", type=int, default=2)
    p.add_argument("--max_length", type=int, default=1024)
    p.add_argument("--lora_r", type=int, default=16)
    p.add_argument("--lora_alpha", type=int, default=32)

    p.add_argument("--num_workers", type=int, default=4)
    p.add_argument("--eval_every", type=int, default=500)
    p.add_argument("--save_every", type=int, default=2000)
    p.add_argument("--eval_frac", type=float, default=0.01)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    p.add_argument("--amp", default="fp16", choices=["fp16", "bf16", "off"])

    a = p.parse_args()
    if a.lr is None:
        a.lr = 1e-3 if a.stage == 1 else 2e-4
    return a


def build_model(args):
    model = TinyVLM(freeze_vision=True, freeze_llm=True, pool_stride=args.pool_stride)

    if args.projector_ckpt:
        sd = torch.load(args.projector_ckpt, map_location="cpu")
        model.projector.load_state_dict(sd)
        print(f"loaded projector from {args.projector_ckpt}")
    elif args.stage == 2:
        print("WARNING: stage 2 without --projector_ckpt -- the projector starts from scratch.")

    model.projector.requires_grad_(True)

    if args.stage == 2:
        from peft import LoraConfig, get_peft_model

        lora_config = LoraConfig(
            r=args.lora_r,
            lora_alpha=args.lora_alpha,
            # The first run only adapted q/k/v/o. The MLP block is where a
            # frozen LLM actually absorbs a new input modality, so it is
            # included here.
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
        )
        model.llm = get_peft_model(model.llm, lora_config)

    return model


@torch.no_grad()
def evaluate(model, loader, device, amp_dtype, max_batches=40):
    model.eval()
    total, n_tok = 0.0, 0
    for i, batch in enumerate(loader):
        if i >= max_batches:
            break
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        with torch.autocast("cuda", dtype=amp_dtype, enabled=amp_dtype is not None):
            out = model(**batch)
        n = int((batch["labels"] != -100).sum())
        if torch.isfinite(out.loss) and n:
            total += out.loss.item() * n
            n_tok += n
    model.train()
    model.vision_encoder.eval()
    return total / max(n_tok, 1)


@torch.no_grad()
def sample_generations(model, examples, image_root, device, amp_dtype, n=2):
    """Generate on a couple of held-out images. If these come out identical for
    different images, the projector has collapsed -- stop and investigate."""
    from PIL import Image

    model.eval()
    placeholder = IMAGE_TOKEN * model.num_visual_tokens
    outs = []
    for ex in examples[:n]:
        img = Image.open(Path(image_root) / ex["image"]).convert("RGB")
        px = model.image_processor(images=img, return_tensors="pt")["pixel_values"].to(device)
        q = ex["conversations"][0]["value"].replace("<image>", "").strip()
        text = build_prompt(model.tokenizer, q, placeholder)
        enc = model.tokenizer(text, return_tensors="pt", add_special_tokens=False)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=amp_dtype is not None):
            ids = model.generate(
                input_ids=enc["input_ids"].to(device),
                attention_mask=enc["attention_mask"].to(device),
                pixel_values=px,
                max_new_tokens=64,
                do_sample=False,
            )
        outs.append((ex["image"], q[:60], model.tokenizer.decode(ids[0], skip_special_tokens=True)))
    model.train()
    model.vision_encoder.eval()
    return outs


def main():
    args = parse_args()
    torch.manual_seed(args.seed)
    random.seed(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)
    print(json.dumps(vars(args), indent=2))

    amp_dtype = {"fp16": torch.float16, "bf16": torch.bfloat16, "off": None}[args.amp]
    if args.device != "cuda":
        amp_dtype = None

    model = build_model(args).to(args.device)
    model.train()
    model.vision_encoder.eval()  # frozen: keep it deterministic
    print(model.trainable_parameter_report())
    print(f"visual tokens per image: {model.num_visual_tokens} (pool_stride={args.pool_stride})")

    full = VLMDataset(
        args.data, args.image_root, model.tokenizer, model.image_processor,
        num_visual_tokens=model.num_visual_tokens,
        image_token=IMAGE_TOKEN, max_length=args.max_length,
    )
    n_eval = max(16, int(len(full) * args.eval_frac))
    idx = list(range(len(full)))
    random.Random(args.seed).shuffle(idx)
    train_ds = Subset(full, idx[n_eval:])
    eval_ds = Subset(full, idx[:n_eval])
    print(f"train {len(train_ds):,} | eval {len(eval_ds):,}")

    collate = make_collate_fn(model.tokenizer.pad_token_id)
    common = dict(collate_fn=collate, num_workers=args.num_workers, pin_memory=True)
    if args.num_workers > 0:
        common.update(persistent_workers=True, prefetch_factor=4)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, drop_last=True, **common)
    eval_loader = DataLoader(eval_ds, batch_size=args.batch_size, shuffle=False, **common)

    sample_examples = [full.examples[i] for i in idx[:2]]

    # --- optimizer: LoRA and an already-trained projector want different LRs ---
    if args.stage == 1:
        groups = [{"params": [p for p in model.projector.parameters() if p.requires_grad], "lr": args.lr}]
    else:
        groups = [
            {"params": [p for p in model.llm.parameters() if p.requires_grad], "lr": args.lr},
            {"params": [p for p in model.projector.parameters() if p.requires_grad], "lr": args.projector_lr},
        ]
    optimizer = torch.optim.AdamW(groups, weight_decay=args.weight_decay, betas=(0.9, 0.95))

    steps_per_epoch = math.ceil(len(train_loader) / args.accum_steps)
    total_steps = steps_per_epoch * args.epochs
    if args.max_steps > 0:
        total_steps = min(total_steps, args.max_steps)
    # Floor of 10 for real runs, but never let warmup eat the whole schedule
    # (matters for short smoke runs, where the LR would otherwise never peak).
    warmup = max(1, min(max(10, int(total_steps * args.warmup_ratio)), total_steps // 5))
    from transformers import get_cosine_schedule_with_warmup
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup, total_steps)
    print(f"optimizer steps: {total_steps:,} (warmup {warmup:,})")

    scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype is torch.float16))

    gstep, micro, running, seen, t0 = 0, 0, 0.0, 0, time.time()
    stop = False
    for epoch in range(args.epochs):
        if stop:
            break
        pbar = tqdm(train_loader, desc=f"stage{args.stage} epoch {epoch}")
        for batch in pbar:
            batch = {k: v.to(args.device, non_blocking=True) for k, v in batch.items()}
            with torch.autocast("cuda", dtype=amp_dtype, enabled=amp_dtype is not None):
                out = model(**batch)
                loss = out.loss

            if not torch.isfinite(loss):
                # a fully-truncated example can yield NaN; drop the micro-batch
                optimizer.zero_grad(set_to_none=True)
                micro = 0
                continue

            scaler.scale(loss / args.accum_steps).backward()
            # report the TRUE loss, not the accumulation-scaled one
            running += loss.item()
            seen += 1
            micro += 1

            if micro == args.accum_steps:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for g in groups for p in g["params"]], args.max_grad_norm
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                gstep += 1
                micro = 0

                pbar.set_postfix(
                    loss=f"{running / max(seen, 1):.3f}",
                    lr=f"{scheduler.get_last_lr()[0]:.2e}",
                    step=gstep,
                )
                if seen >= 50:
                    running, seen = 0.0, 0

                if gstep % args.eval_every == 0:
                    ev = evaluate(model, eval_loader, args.device, amp_dtype)
                    gens = sample_generations(model, sample_examples, args.image_root, args.device, amp_dtype)
                    tqdm.write(f"\n[step {gstep}] eval_loss={ev:.4f}  elapsed={(time.time()-t0)/60:.1f}m")
                    for img, q, g in gens:
                        tqdm.write(f"   {img} | {q!r}\n     -> {g[:200]!r}")
                    if len({g for _, _, g in gens}) == 1 and len(gens) > 1:
                        tqdm.write("   !! identical outputs on different images -- projector may be collapsing")

                if args.save_every > 0 and gstep % args.save_every == 0:
                    save(model, args, tag=f"step{gstep}")

                if args.max_steps > 0 and gstep >= args.max_steps:
                    stop = True
                    break

    ev = evaluate(model, eval_loader, args.device, amp_dtype)
    print(f"\nfinal eval_loss = {ev:.4f}  ({(time.time()-t0)/60:.1f} min, {gstep} steps)")
    save(model, args)
    (Path(args.output_dir) / "train_meta.json").write_text(
        json.dumps({**vars(args), "final_eval_loss": ev, "steps": gstep,
                    "num_visual_tokens": model.num_visual_tokens}, indent=2)
    )


def save(model, args, tag=None):
    out = Path(args.output_dir) if tag is None else Path(args.output_dir) / tag
    out.mkdir(parents=True, exist_ok=True)
    torch.save(model.projector.state_dict(), out / "projector.pt")
    if args.stage == 2:
        model.llm.save_pretrained(out / "lora_adapter")
    print(f"saved -> {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile inference.py
"""
inference.py

Load base models + trained projector + LoRA adapter and answer questions
about an image.

Two things here are load-bearing:

*   The prompt is built by `dataset.build_prompt`, the same function the
    training data used. In the first version the training code and the
    inference code each rendered the chat template their own way; they happened
    to agree for Qwen, but nothing enforced it.

*   `pool_stride` is recovered from the projector checkpoint's own shape rather
    than passed in. A mismatch here silently changes how many visual tokens the
    model expects and produces garbage, so it should not be a flag a human can
    get wrong.

Usage:
    python inference.py --image cat.jpg --question "What is in this image?" \
        --projector_ckpt ckpt/stage2/projector.pt --lora_dir ckpt/stage2/lora_adapter
"""

import argparse
import sys
from pathlib import Path

import torch
from PIL import Image

for _p in (Path(__file__).resolve().parent.parent / "Model", Path(__file__).resolve().parent):
    sys.path.insert(0, str(_p))

from model import TinyVLM, IMAGE_TOKEN            # noqa: E402
from dataset import build_prompt                  # noqa: E402


def infer_pool_stride(state_dict, vision_dim: int = 768) -> int:
    """The projector's first Linear has in_features = vision_dim * stride^2."""
    w = state_dict.get("net.0.weight")
    if w is None:
        return 2
    stride = round((w.shape[1] / vision_dim) ** 0.5)
    if vision_dim * stride * stride != w.shape[1]:
        raise ValueError(
            f"cannot derive pool_stride: projector expects {w.shape[1]} input features, "
            f"which is not vision_dim({vision_dim}) * k^2"
        )
    return stride


def load_model(projector_ckpt: str, lora_dir: str = None, device: str = "cuda", dtype=torch.float32):
    sd = torch.load(projector_ckpt, map_location="cpu")
    pool_stride = infer_pool_stride(sd)

    model = TinyVLM(freeze_vision=True, freeze_llm=True, pool_stride=pool_stride, dtype=dtype)
    model.projector.load_state_dict(sd)

    if lora_dir:
        from peft import PeftModel
        model.llm = PeftModel.from_pretrained(model.llm, lora_dir)
        model.llm = model.llm.merge_and_unload()   # fold LoRA in; faster inference

    model = model.to(device).eval()
    print(f"loaded: pool_stride={pool_stride}, visual tokens={model.num_visual_tokens}")
    return model


@torch.no_grad()
def answer(
    model,
    image,
    question: str,
    device: str = "cuda",
    max_new_tokens: int = 128,
    short_answer: bool = False,
    temperature: float = 0.0,
    repetition_penalty: float = 1.05,
) -> str:
    """`image` may be a path or a PIL.Image."""
    if not isinstance(image, Image.Image):
        image = Image.open(image)
    image = image.convert("RGB")

    pixel_values = model.image_processor(images=image, return_tensors="pt")["pixel_values"].to(device)
    placeholder = IMAGE_TOKEN * model.num_visual_tokens
    text = build_prompt(model.tokenizer, question, placeholder, short_answer=short_answer)
    enc = model.tokenizer(text, return_tensors="pt", add_special_tokens=False)

    gen = dict(
        max_new_tokens=16 if short_answer else max_new_tokens,
        repetition_penalty=repetition_penalty,
    )
    if temperature and temperature > 0:
        gen.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen.update(do_sample=False)

    out = model.generate(
        input_ids=enc["input_ids"].to(device),
        attention_mask=enc["attention_mask"].to(device),
        pixel_values=pixel_values,
        **gen,
    )
    # inputs_embeds (not input_ids) went into generate(), so HF returns only
    # the newly generated tokens.
    return model.tokenizer.decode(out[0], skip_special_tokens=True).strip()


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--image", required=True)
    p.add_argument("--question", required=True)
    p.add_argument("--projector_ckpt", required=True)
    p.add_argument("--lora_dir", default=None)
    p.add_argument("--short", action="store_true", help="append the short-answer hint (VQA mode)")
    p.add_argument("--max_new_tokens", type=int, default=128)
    p.add_argument("--temperature", type=float, default=0.0)
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = p.parse_args()

    m = load_model(a.projector_ckpt, a.lora_dir, a.device)
    print(answer(m, a.image, a.question, a.device, a.max_new_tokens,
                 short_answer=a.short, temperature=a.temperature))

In [ ]:
%%writefile diagnose_grounding.py
"""
diagnose_grounding.py

The check that would have caught the original failure before it ever reached a
Space: run the SAME question against several DIFFERENT images and compare.

If the outputs are identical across genuinely different images, the LLM is
answering from the question prior alone and the projector carries no usable
signal. No amount of decoding-parameter tuning fixes that -- it means the
alignment stage did not work and has to be redone.

Usage:
    python diagnose_grounding.py \
        --projector_ckpt ckpt/stage2/projector.pt \
        --lora_dir ckpt/stage2/lora_adapter \
        --images a.jpg b.jpg c.jpg
"""

import argparse
import sys
from pathlib import Path

import torch
from PIL import Image

for _p in (Path(__file__).resolve().parent.parent / "Model", Path(__file__).resolve().parent):
    sys.path.insert(0, str(_p))

from inference import load_model, answer   # noqa: E402


@torch.no_grad()
def visual_tokens(model, image, device):
    px = model.image_processor(images=image.convert("RGB"), return_tensors="pt")["pixel_values"].to(device)
    return model.encode_image(px)[0].float()   # [num_visual_tokens, llm_dim]


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--projector_ckpt", required=True)
    p.add_argument("--lora_dir", default=None)
    p.add_argument("--images", nargs="+", required=True)
    p.add_argument("--question", default="What is in this image?")
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = p.parse_args()

    model = load_model(a.projector_ckpt, a.lora_dir, a.device)
    images = [Image.open(i).convert("RGB") for i in a.images]
    names = [Path(i).name for i in a.images]

    blank = Image.new("RGB", (224, 224), (0, 0, 0))
    noise = Image.effect_noise((224, 224), 64).convert("RGB")

    print("=" * 74)
    print("TEST 1 -- same question, different images")
    print("=" * 74)
    outs = []
    for name, img in list(zip(names, images)) + [("<blank>", blank), ("<noise>", noise)]:
        long_a = answer(model, img, a.question, a.device, short_answer=False)
        short_a = answer(model, img, a.question, a.device, short_answer=True)
        outs.append((name, long_a, short_a))
        print(f"  {name:24s} long : {long_a[:120]!r}")
        print(f"  {'':24s} short: {short_a[:60]!r}")

    real_long = [o[1] for o in outs[: len(images)]]
    real_short = [o[2] for o in outs[: len(images)]]
    print(f"\n  distinct long answers  : {len(set(real_long))} / {len(real_long)}")
    print(f"  distinct short answers : {len(set(real_short))} / {len(real_short)}")
    if len(set(real_long)) == 1:
        print("  >>> VERDICT: image completely ignored (pure language prior).")
    elif len(set(real_long)) < len(real_long):
        print("  >>> VERDICT: weak grounding -- some images collapse to the same answer.")
    else:
        print("  >>> Outputs vary with the image. Grounding is working.")

    avg_len = sum(len(o[1].split()) for o in outs[: len(images)]) / max(len(images), 1)
    print(f"  mean long-answer length: {avg_len:.1f} words "
          f"({'TOO SHORT -- collapsed to VQA style' if avg_len < 4 else 'ok'})")

    print()
    print("=" * 74)
    print("TEST 2 -- projector output scale vs. LLM embedding scale")
    print("=" * 74)
    vs = [visual_tokens(model, im, a.device) for im in images]
    emb = model.llm.get_input_embeddings().weight
    emb = emb[: len(model.tokenizer)].float()
    v_norm = vs[0].norm(dim=-1).mean().item()
    t_norm = emb.norm(dim=-1).mean().item()
    print(f"  visual tokens per image : {vs[0].shape[0]}")
    print(f"  visual token  std / norm: {vs[0].std().item():.4f} / {v_norm:.3f}")
    print(f"  text embedding std / norm: {emb.std().item():.4f} / {t_norm:.3f}")
    print(f"  norm ratio (visual/text): {v_norm / t_norm:.2f}x")
    if not 0.33 < v_norm / t_norm < 3.0:
        print("  >>> Visual tokens sit outside the LLM's embedding distribution.")

    print()
    print("=" * 74)
    print("TEST 3 -- do different images produce different visual tokens?")
    print("=" * 74)
    # Deliberately NOT cosine over mean-pooled tokens. Projected visual tokens
    # share a large constant component, so pooled cosine sits at ~0.99 even for
    # a perfectly well-grounded model -- it reports collapse that isn't there.
    # Relative L2 distance over the full token matrix is the honest measure:
    # 0 = identical, ~1.4 = unrelated.
    V = torch.stack([v.flatten() for v in vs])
    rels = []
    for i in range(len(vs)):
        for j in range(i + 1, len(vs)):
            denom = (V[i].norm() + V[j].norm()) / 2
            rel = ((V[i] - V[j]).norm() / denom).item()
            rels.append(rel)
            print(f"  rel_dist({names[i]}, {names[j]}) = {rel:.4f}")

    varying = (V - V.mean(0)).norm() / V.norm()
    print(f"\n  image-dependent fraction of the representation: {varying:.4f}")
    if rels and min(rels) < 0.02:
        print("  >>> Projector has collapsed to a near-constant output.")
    elif rels:
        print(f"  min rel_dist = {min(rels):.4f} -- projector output varies with the image.")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile push_to_hub.py
"""
push_to_hub.py

Uploads the trained projector + LoRA adapter + a model card.

The adapter is now ~35 MB rather than 1.1 GB: the first version added an
`<image>` token and called `resize_token_embeddings`, which made PEFT serialise
the entire 151k x 896 embedding matrix into the adapter. Reusing Qwen's
existing `<|image_pad|>` token removes that entirely.

Usage:
    python push_to_hub.py --repo_id you/tinyvlm --projector_ckpt ckpt/stage2/projector.pt \
        --lora_dir ckpt/stage2/lora_adapter
"""

import argparse
import json
import shutil
import tempfile
from pathlib import Path

from huggingface_hub import HfApi, create_repo

CARD = """---
license: apache-2.0
base_model: Qwen/Qwen2.5-0.5B-Instruct
tags: [vision-language-model, vqa, image-captioning, lora, siglip2, qwen2.5]
---

# {repo_name}

A compact LLaVA-style VLM: SigLIP2 vision encoder -> pixel-shuffle -> MLP
projector -> Qwen2.5-0.5B-Instruct with LoRA.

| | |
|---|---|
| Vision encoder | `google/siglip2-base-patch16-224` (frozen) |
| LLM | `Qwen/Qwen2.5-0.5B-Instruct` (LoRA: q,k,v,o,gate,up,down) |
| Visual tokens | {num_visual_tokens} per image (14x14 patches, 2x2 pixel-shuffle) |
| Image placeholder | `<|image_pad|>` -- an existing Qwen token, so no vocab resize |
| Stage 1 | projector alignment on Flickr8k + Flickr30k captions |
| Stage 2 | LoRA instruction tuning on VQAv2 + LLaVA-ReCap + captions |

## Answer length is prompt-controlled

Stage 2 mixes one-word VQA answers with multi-sentence descriptions, and VQA
examples carry an explicit hint. Append it for short answers, omit it for prose:

```
What color is the bus?
Answer the question using a single word or phrase.     -> "red"

What color is the bus?                                 -> "The bus is red."
```

## Files

- `projector.pt` -- projector weights (`pool_stride` is recoverable from the tensor shapes)
- `lora_adapter/` -- PEFT adapter for the LLM

## Usage

Needs the companion code from the training repo (`Model/model.py`,
`Model/dataset.py`, `Inference/inference.py`):

```python
from huggingface_hub import hf_hub_download, snapshot_download
from inference import load_model, answer

projector = hf_hub_download(repo_id="{repo_id}", filename="projector.pt")
lora = snapshot_download(repo_id="{repo_id}", allow_patterns=["lora_adapter/*"]) + "/lora_adapter"

model = load_model(projector, lora, device="cuda")
print(answer(model, "photo.jpg", "What is in this image?"))
print(answer(model, "photo.jpg", "What color is the car?", short_answer=True))
```

## Limitations

224x224 input, {num_visual_tokens} visual tokens, and a 0.5B LLM. Expect
everyday-scene captioning and simple VQA -- not OCR, fine detail, counting, or
multi-step visual reasoning.

{metrics}
"""


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--repo_id", required=True)
    p.add_argument("--projector_ckpt", required=True)
    p.add_argument("--lora_dir", required=True)
    p.add_argument("--meta", default=None, help="train_meta.json from stage 2")
    p.add_argument("--private", action="store_true")
    a = p.parse_args()

    num_visual_tokens, metrics = 49, ""
    if a.meta and Path(a.meta).exists():
        meta = json.loads(Path(a.meta).read_text())
        num_visual_tokens = meta.get("num_visual_tokens", 49)
        metrics = (
            "## Training\n\n"
            f"- optimizer steps: {meta.get('steps')}\n"
            f"- final held-out loss: {meta.get('final_eval_loss'):.4f}\n"
            f"- batch size {meta.get('batch_size')} x {meta.get('accum_steps')} accum, "
            f"lr {meta.get('lr')}\n"
        )

    api = HfApi()
    create_repo(a.repo_id, private=a.private, exist_ok=True)

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        shutil.copy(a.projector_ckpt, tmp / "projector.pt")
        shutil.copytree(a.lora_dir, tmp / "lora_adapter")
        if a.meta and Path(a.meta).exists():
            shutil.copy(a.meta, tmp / "train_meta.json")
        (tmp / "README.md").write_text(
            CARD.format(
                repo_name=a.repo_id.split("/")[-1],
                repo_id=a.repo_id,
                num_visual_tokens=num_visual_tokens,
                metrics=metrics,
            ),
            encoding="utf-8",
        )
        api.upload_folder(folder_path=str(tmp), repo_id=a.repo_id,
                          commit_message="TinyVLM projector + LoRA adapter")

    print(f"https://huggingface.co/{a.repo_id}")


if __name__ == "__main__":
    main()

In [ ]:
!ls -la /kaggle/working/*.py

## 3. Data

Four corpora, all written into one shared image directory as JPEGs resized to a
256px short side. Run 1 symlinked full-resolution originals and re-decoded them
every epoch.

| corpus | role | rows |
|---|---|---|
| Flickr8k | alignment captions | ~40k (8k images) |
| Flickr30k | alignment captions | ~145k (29k images) |
| VQAv2 | short answers | 40k |
| LLaVA-ReCap-118K | detailed descriptions | 25k |

In [ ]:
!ls /kaggle/input

In [ ]:
# Flickr8k, from the Kaggle input attached in the sidebar. If this fails, the
# traceback prints the directory tree it searched so you can point it correctly.
!python prepare_data.py --task flickr8k \
    --kaggle_input_dir /kaggle/input \
    --out_jsonl /kaggle/working/data/cap8k.jsonl \
    --out_image_dir /kaggle/working/data/images

In [ ]:
# ~29k more distinct images. This is the single biggest change from run 1:
# the projector now sees ~24x more visual variety.
!python prepare_data.py --task flickr30k \
    --out_jsonl /kaggle/working/data/cap30k.jsonl \
    --out_image_dir /kaggle/working/data/images

In [ ]:
!python prepare_data.py --task vqav2 --max_examples 40000 \
    --out_jsonl /kaggle/working/data/vqa.jsonl \
    --out_image_dir /kaggle/working/data/images

In [ ]:
# Detailed multi-sentence descriptions -- what restores the ability to write
# prose rather than a single word.
!python prepare_data.py --task recap --max_examples 25000 \
    --out_jsonl /kaggle/working/data/recap.jsonl \
    --out_image_dir /kaggle/working/data/images

In [ ]:
# Stage 1 = pure captioning. Stage 2 = a deliberate mix of answer styles.
!python prepare_data.py --task mix \
    --inputs /kaggle/working/data/cap8k.jsonl /kaggle/working/data/cap30k.jsonl \
    --out_jsonl /kaggle/working/data/stage1.jsonl

!python prepare_data.py --task mix \
    --inputs /kaggle/working/data/vqa.jsonl /kaggle/working/data/recap.jsonl /kaggle/working/data/cap8k.jsonl \
    --caps 40000 25000 20000 \
    --out_jsonl /kaggle/working/data/stage2.jsonl

In [ ]:
import json, collections
for name in ["stage1", "stage2"]:
    rows = [json.loads(l) for l in open(f"/kaggle/working/data/{name}.jsonl", encoding="utf-8")]
    styles = collections.Counter(r.get("style", "caption") for r in rows)
    ans = [len(t["value"].split()) for r in rows for t in r["conversations"] if t["from"] == "gpt"]
    print(f"{name}: {len(rows):,} rows | styles {dict(styles)}")
    print(f"   answer length: mean {sum(ans)/len(ans):.1f} words, min {min(ans)}, max {max(ans)}")

## 4. Stage 1 -- projector alignment

Vision encoder and LLM frozen; only the projector trains. Gradients still flow
*through* the LLM to reach it, so this is not cheap, but no LLM weights move.

Watch the sample generations printed every 500 steps. If two different images
produce the same caption, stop -- Stage 2 will not rescue it.

In [ ]:
!python train.py --stage 1 \
    --data /kaggle/working/data/stage1.jsonl \
    --image_root /kaggle/working/data/images \
    --output_dir /kaggle/working/ckpt/stage1 \
    --epochs 2 --batch_size 32 --accum_steps 1 --lr 1e-3 \
    --num_workers 4 --eval_every 500 --save_every 4000

### Grounding gate

Run this before spending GPU time on Stage 2. Three visually different images,
one question. Identical answers means the projector learned nothing, and no
amount of Stage-2 tuning or decoding-parameter fiddling fixes that.

In [ ]:
import json
rows = [json.loads(l) for l in open("/kaggle/working/data/stage2.jsonl", encoding="utf-8")][:400]
seen, picks = set(), []
for r in rows:                      # one image from each source corpus
    tag = r["image"].split("_")[0]
    if tag not in seen:
        seen.add(tag); picks.append("/kaggle/working/data/images/" + r["image"])
    if len(picks) == 3:
        break
IMGS = " ".join(picks)
print(IMGS)

In [ ]:
!python diagnose_grounding.py \
    --projector_ckpt /kaggle/working/ckpt/stage1/projector.pt \
    --images {IMGS}

## 5. Stage 2 -- LoRA instruction tuning

LoRA on attention **and** MLP projections; the projector keeps training at a much
lower LR so it is refined rather than overwritten.

In [ ]:
!python train.py --stage 2 \
    --data /kaggle/working/data/stage2.jsonl \
    --image_root /kaggle/working/data/images \
    --projector_ckpt /kaggle/working/ckpt/stage1/projector.pt \
    --output_dir /kaggle/working/ckpt/stage2 \
    --epochs 3 --batch_size 16 --accum_steps 1 \
    --lr 2e-4 --projector_lr 2e-5 --lora_r 16 --lora_alpha 32 \
    --num_workers 4 --eval_every 500 --save_every 4000

### Grounding gate again, now with the adapter

In [ ]:
!python diagnose_grounding.py \
    --projector_ckpt /kaggle/working/ckpt/stage2/projector.pt \
    --lora_dir /kaggle/working/ckpt/stage2/lora_adapter \
    --images {IMGS}

## 6. Qualitative check

Both answer styles on the same images. Descriptive prompts should produce
sentences; the short-answer hint should produce a word or two. If everything
comes back as one word, the Stage-2 style mix did not take.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")
from inference import load_model, answer
from PIL import Image
import matplotlib.pyplot as plt

model = load_model("/kaggle/working/ckpt/stage2/projector.pt",
                   "/kaggle/working/ckpt/stage2/lora_adapter", "cuda")

fig, axes = plt.subplots(1, len(picks), figsize=(5 * len(picks), 5))
for ax, path in zip(axes if len(picks) > 1 else [axes], picks):
    img = Image.open(path).convert("RGB")
    long_a = answer(model, img, "What is in this image?", "cuda")
    short_a = answer(model, img, "What is in this image?", "cuda", short_answer=True)
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"long: {long_a[:70]}\nshort: {short_a[:40]}", fontsize=8, wrap=True)
    print(f"{path}\n  descriptive: {long_a}\n  short      : {short_a}\n")
plt.tight_layout(); plt.show()

In [ ]:
# Exact-match VQA accuracy on the short-answer prompt.
import json, random
rows = [json.loads(l) for l in open("/kaggle/working/data/vqa.jsonl", encoding="utf-8")]
random.Random(1).shuffle(rows)

hits = 0
sample = rows[:150]
for r in sample:
    q = r["conversations"][0]["value"].replace("<image>", "").split("\nAnswer the question")[0].strip()
    gt = r["conversations"][1]["value"].strip().lower()
    pred = answer(model, "/kaggle/working/data/images/" + r["image"], q, "cuda",
                  short_answer=True).strip().lower().rstrip(".")
    hits += (pred == gt)
print(f"exact match on {len(sample)} rows: {hits/len(sample):.1%}")
print("(a question-prior-only model lands near 25-30%; grounded should be clearly above)")

## 7. Push weights to the Hub

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("No Kaggle secret named HF_TOKEN -- add one under Add-ons > Secrets.")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"]) if "HF_TOKEN" in os.environ else login()

In [ ]:
!python push_to_hub.py \
    --repo_id dhruvpatel93/tinyvlm-vqa \
    --projector_ckpt /kaggle/working/ckpt/stage2/projector.pt \
    --lora_dir /kaggle/working/ckpt/stage2/lora_adapter \
    --meta /kaggle/working/ckpt/stage2/train_meta.json

## 8. The Space does NOT update itself

Pushing weights to the model repo does not rebuild a Space, and restarting the
existing one by hand will crash: it runs run-1 code, and these checkpoints are
not loadable by it. The projector gained an `out_norm`, and its first Linear
went from 768 to 3072 input features (2x2 pixel-shuffle), so `load_state_dict`
raises immediately.

Deploy the code and the weights together, from the training repo:

```bash
python deploy_space.py --space_id <you>/tinyvlm --model_repo dhruvpatel93/tinyvlm-vqa
```

That flattens `model.py`, `dataset.py`, `inference.py` and `app_gradio.py` into
the Space, sets `HF_REPO_ID` so the app pulls weights from the Hub at startup,
and triggers a rebuild. After that first deploy, weights-only changes need just
a push plus a Space restart.